In [2]:
from dotenv import load_dotenv
load_dotenv()

import os
from pprint import pprint

from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

In [3]:
llm = OpenAI(model="gpt-4o", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

Settings.llm = llm

In [4]:
from tavily import TavilyClient
from llama_index.core.tools import FunctionTool

def tavily_search(query: str):
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    return client.search(query=query, search_depth="basic", max_results=5)

search_tool = FunctionTool.from_defaults(fn=tavily_search, name="web_search", description="Search the web for recent or unknown information.")

In [7]:
result = tavily_search("كم سعر الدينار الليبي مقابل الدولار اليوم في ليبيا؟")
pprint(result)

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'كم سعر الدينار الليبي مقابل الدولار اليوم في ليبيا؟',
 'request_id': '2bce0051-d69e-48ce-b4f3-4f6ab84ca2ec',
 'response_time': 3.6,
 'results': [{'content': 'Rating 4.0 (40) اليوم الاربعاء 04 - 03 - 2026. سعر '
                         'الدولار بمصرف ليبيا المركزي لهذا اليوم: سعر البيع: '
                         '6.3917 دينار. سعر الشراء: 6.3598 دينار. - اسعار '
                         'العملات في',
              'raw_content': None,
              'score': 0.9106172,
              'title': 'اسعار العملات في ليبيا في السوق السوداء '
                       '(@libyacurrencyprices)',
              'url': 'https://www.facebook.com/libyacurrencyprices/'},
             {'content': 'Rating 4.8 (105,053) حوِّل الدينار الليبي إلى '
                         'الدولار الأمريكي ; ١ LYD, ٠٫١٥٧٤٠٢ USD ; ٥ LYD, '
                         '٠٫٧٨٧٠١ USD ; ١٠ LYD, ١٫٥٧٤٠٢ USD ; ٢٥ LYD, ٣٫٩٣٥٠٥ '
                         'USD.',
    

In [10]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context

agent = ReActAgent(
    tools=[search_tool],
    llm=llm,
    verbose=True,
)

ctx = Context(agent)

In [13]:
handler = await agent.run(
    "سعر الدينار الليبي مقابل الدولار في السوق الموازي اليوم في ليبيا؟",
    ctx=ctx
)
print(handler.response)

assistant: سعر الدينار الليبي مقابل الدولار في السوق الموازي اليوم هو حوالي 10.23 دينار ليبي لكل دولار أمريكي.
